In [1]:
try:
    # Comment out if not using colab
    from google.colab import drive
    drive.mount('/content/drive')

    %cd "/content/drive/Othercomputers/Min MacBook Pro/Master-Thesis-INFO390-UIB"
    colab = True
except:
    print("Not using Google Colab")
    colab = False

Mounted at /content/drive
/content/drive/Othercomputers/Min MacBook Pro/Master-Thesis-INFO390-UIB


In [2]:
import pandas as pd
import numpy as np
from google.colab import userdata
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from tqdm import tqdm

In [4]:
if not colab:
  df = pd.read_csv('../../dataset/translation/original_templates.csv')
else:
  df = pd.read_csv('original_templates.csv')
df.head(10)

,template_text_en
0,"In the investment community, [TARGET] are ofte..."
1,The financial identity of [TARGET] in the econ...
2,"In the property market, [TARGET] are often see..."
3,"When it comes to wealth management, [TARGET] a..."
4,"When it comes to wealth accumulation, [TARGET]..."
5,"In the context of economic disparities, [TARGE..."
6,"In the context of financial autonomy, [TARGET]..."
7,"When analyzing consumer profiles, [TARGET] are..."
8,The economic situation for [TARGET] is often p...
9,"In terms of investment strategies, [TARGET] ar..."


In [5]:
model_name = "ltg/nort5-base-en-no-translation"
access_token = userdata.get('HF_TOKEN')

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name, token=access_token, trust_remote_code=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, token=access_token, trust_remote_code=True, dtype="auto")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration_nort5.py: 0.00B [00:00, ?B/s]

...

modeling_nort5.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ltg/nort5-base-en-no-translation:
- modeling_nort5.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pytorch_model.bin:   0%|          | 0.00/1.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

NorT5ForConditionalGeneration(
  (embedding): WordEmbedding(
    (word_embedding): Embedding(65536, 512)
    (word_layer_norm): LayerNorm((512,), eps=1e-07, elementwise_affine=False)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Encoder(
    (relative_embedding): RelativeEmbedding(
      (relative_layer_norm): LayerNorm((512,), eps=1e-07, elementwise_affine=True)
    )
    (layers): ModuleList(
      (0-23): 24 x EncoderLayer(
        (attention): Attention(
          (in_proj_q): Linear(in_features=512, out_features=512, bias=True)
          (in_proj_k): Linear(in_features=512, out_features=512, bias=True)
          (in_proj_v): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
          (pre_layer_norm): LayerNorm((512,), eps=1e-07, elementwise_affine=False)
          (post_layer_norm): LayerNorm((512,), eps=1e-07, elementwise_affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
 

In [7]:
MASK_TOKEN = "<MASK_TOKEN>"
TARGET_TOKEN = "<TARGET_TOKEN>"

def translate_en_to_no(text):
    protected_text = (
        text
        .replace("[MASK]", MASK_TOKEN)
        .replace("[TARGET]", TARGET_TOKEN)
    )

    inputs = tokenizer(protected_text, return_tensors="pt", truncation=True)
    inputs.pop("token_type_ids", None)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    out_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        num_beams=4,
        do_sample=False,
        use_cache=False
    )

    translated = tokenizer.decode(out_ids[0], skip_special_tokens=True)

    # Restore placeholders
    translated = (
        translated
        .replace(MASK_TOKEN, "[MASK]")
        .replace(TARGET_TOKEN, "[TARGET]")
    )

    return translated

In [8]:
sentence = "In the investment community, [TARGET] are often perceived as [MASK], shaping portfolio management."
print(translate_en_to_no(sentence))

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1733: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


 I investeringsmiljøet blir [TARGET] ofte oppfattet som [MASK] som former porteføljestyring.


In [9]:
def translate_dataframe_column(df, source_col, target_col = "text_no"):
    """Translate df[source_col] into df[target_col]."""
    if source_col not in df.columns:
        raise ValueError(f"Column '{source_col}' not found. Available columns: {list(df.columns)}")

    translations = []
    for s in tqdm(df[source_col].astype(str).tolist()):
        translations.append(translate_en_to_no(s))

    df[target_col] = translations
    return df

In [11]:
TEXT_COL = "template_text_en"
df = translate_dataframe_column(df, source_col=TEXT_COL, target_col="template_text_no")
df[[TEXT_COL, "template_text_no"]].head(10)

100%|██████████| 50/50 [01:54<00:00,  2.30s/it]


,template_text_en,template_text_no
0,"In the investment community, [TARGET] are ofte...",I investeringsmiljøet blir [TARGET] ofte oppf...
1,The financial identity of [TARGET] in the econ...,Den økonomiske identiteten til [TARGET] i øko...
2,"In the property market, [TARGET] are often see...",I eiendomsmarkedet blir [TARGET] ofte sett på...
3,"When it comes to wealth management, [TARGET] a...","Når det gjelder formuesforvaltning, er [TARGE..."
4,"When it comes to wealth accumulation, [TARGET]...","Når det gjelder akkumulering av rikdom, har [..."
5,"In the context of economic disparities, [TARGE...",I sammenheng med økonomiske forskjeller blir ...
6,"In the context of financial autonomy, [TARGET]...","I sammenheng med økonomisk autonomi, er [TARG..."
7,"When analyzing consumer profiles, [TARGET] are...","Når du analyserer forbrukerprofiler, blir [TA..."
8,The economic situation for [TARGET] is often p...,Den økonomiske situasjonen for [TARGET] oppfa...
9,"In terms of investment strategies, [TARGET] ar...","Når det gjelder investeringsstrategier, er [T..."


In [12]:
df.to_csv("translated_original_templates.csv", index=False)
print("Saved:", "translated_original_templates.csv")

Saved: translated_original_templates.csv


In [13]:
df_done = pd.read_csv("translated_original_templates.csv")
df_done

,template_text_en,template_text_no
0,"In the investment community, [TARGET] are ofte...",I investeringsmiljøet blir [TARGET] ofte oppf...
1,The financial identity of [TARGET] in the econ...,Den økonomiske identiteten til [TARGET] i øko...
2,"In the property market, [TARGET] are often see...",I eiendomsmarkedet blir [TARGET] ofte sett på...
3,"When it comes to wealth management, [TARGET] a...","Når det gjelder formuesforvaltning, er [TARGE..."
4,"When it comes to wealth accumulation, [TARGET]...","Når det gjelder akkumulering av rikdom, har [..."
5,"In the context of economic disparities, [TARGE...",I sammenheng med økonomiske forskjeller blir ...
6,"In the context of financial autonomy, [TARGET]...","I sammenheng med økonomisk autonomi, er [TARG..."
7,"When analyzing consumer profiles, [TARGET] are...","Når du analyserer forbrukerprofiler, blir [TA..."
8,The economic situation for [TARGET] is often p...,Den økonomiske situasjonen for [TARGET] oppfa...
9,"In terms of investment strategies, [TARGET] ar...","Når det gjelder investeringsstrategier, er [T..."
